In [ ]:
import pandas as pd
from pathlib import Path 
import geopandas as geopd
import plotly.graph_objs as go 
import plotly.figure_factory as ff
import plotly.express as px  
import matplotlib.pyplot as plt 
import numpy as np 

from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.metrics import silhouette_score
from scipy.signal import find_peaks

import sys
sys.path.insert(0, str(Path("/home/aricci/stampa_comuni_trentini").resolve()))
from utils import get_s3, get_mapping
from aixpa_analisi_indici.help_plots import plot_stats

# Download dei dati e preprocessing

In [ ]:
vodafone_attendences = pd.read_csv(get_s3("vodafone_attendences.csv"))
vodafone_attendences_new = pd.read_csv(get_s3("vodafone_attendences_new.csv"))

geojson_comuni_json_data = geopd.read_file(get_s3("TRENTINO-comuni_Vodafone_2023.geojson"))
json_vodafone = get_mapping("mapping_comuni_into_vodafone_Trento.json")
json_apt = get_mapping("map_comuni_into_apt.json")

In [ ]:
location_map = dict(
    zip(
        geojson_comuni_json_data["id"].astype(str),
        geojson_comuni_json_data["name"].str.upper(),
    )
)

vodafone_attendences['date'] = pd.to_datetime(vodafone_attendences['date'])
vodafone_attendences_new['date'] = pd.to_datetime(vodafone_attendences_new['date'])
vodafone_attendences = vodafone_attendences[~vodafone_attendences['date'].isin(vodafone_attendences_new['date'])]

# Merge
vodafone_attendences_merged = pd.concat([vodafone_attendences, vodafone_attendences_new], ignore_index=True)
vodafone_attendences_merged = vodafone_attendences_merged.sort_values('date').reset_index(drop=True)

vodafone_attendences_merged_comuni = vodafone_attendences_merged[
    vodafone_attendences_merged["locType"] == "TN_MKT_AL_3"
]

vodafone_attendences_merged_comuni["comune"] = vodafone_attendences_merged_comuni["locId"].astype(str).map(location_map)
vodafone_attendences_merged_comuni["ID_COMUNE"] = vodafone_attendences_merged_comuni["comune"].map(json_vodafone)
mask = vodafone_attendences_merged_comuni["comune"].isin(["VIGO DI FASSA", "POZZA DI FASSA"])
vodafone_attendences_merged_comuni.loc[mask, "comune"] = "SAN GIOVANNI DI FASSA" 
vodafone_attendences_merged_comuni.loc[mask, "ID_COMUNE"] = pd.Series(
    [[22250]] * mask.sum(), index=vodafone_attendences_merged_comuni.index[mask]  
)

tourists_attendences = vodafone_attendences_merged_comuni[
    vodafone_attendences_merged_comuni['userProfile'] == 'TOURIST'
].groupby(['date', 'comune']).agg({'value': 'sum', 'ID_COMUNE':'first'}).reset_index()

# SETTING PARAMETERS

In [ ]:
ROLLING = 14 
PROMINCENCE = 0.3
DISTANCE = 60
DATE_RANGE = pd.date_range("2022-01-01", "2025-12-31", freq="D") ## CUT DATES 
CLUSTERS = 3
METHOD = "ward"
FORMAT_SAVE = "parquet"

def ward_distance(corr_matrix):
    return np.sqrt(2 * (1 - corr_matrix)) 

# 1. ROLLING WINDOW + 2. NORMALIZZAZIONE 

In [ ]:
# PIVOT MATRIX 
s = (
    tourists_attendences.pivot(index="date", columns="comune", values="value")
    .reindex(DATE_RANGE)
    .fillna(0)
)

# ROLLING DI 14 
ts_rolling_14 = s.rolling(window=ROLLING, center=True).mean()
min_vals = ts_rolling_14.min()
max_vals = ts_rolling_14.max()
denom = max_vals - min_vals

# 2. NORMALIZZAZIONE (Fatti in anticipo su tutti i comuni)
ts_normalized = (ts_rolling_14 - min_vals) / denom.replace(0, np.nan)
ts_normalized = ts_normalized.fillna(0)

# CORRELAZIONE 
corr_matrix = ts_normalized.corr()
corr_matrix

# 3. CLUSTERING

In [ ]:
dist_matrix = ward_distance(corr_matrix) if METHOD == "ward" else 1 - corr_matrix
condensed_dist = squareform(dist_matrix.values, checks=False)
Z = linkage(condensed_dist, method=METHOD)

dendogram = ff.create_dendrogram(
    dist_matrix.values,
    labels=corr_matrix.columns.tolist(),
    distfun=lambda x: condensed_dist,
    linkagefun=lambda d: Z,
)


dendogram.update_layout(
    title=f"Dendrogramma della Similarità Turistica (su dati Rolling + Normalizzati), metodo {METHOD}",
    xaxis_title="Comuni",
    yaxis_title="Distanza",
    width=2000,
    height=700,
    template="plotly_white"
)
dendogram.update_xaxes(tickangle=-45)
dendogram.show()

clusters = fcluster(Z, t=CLUSTERS, criterion='maxclust')

df_clusters = pd.DataFrame({
    'comune': corr_matrix.columns,
    'cluster': clusters
}).sort_values('cluster')

print(f"--- Distribuzione Comuni nei {CLUSTERS} Cluster ---")
print(df_clusters['cluster'].value_counts())

In [ ]:
print("--- Confronto Silhouette Score ---")
for method in ["average", "complete", "single", "ward"]:
    Z_test = linkage(condensed_dist, method=method)
    
    for k in range(2, 8):
        labels = fcluster(Z_test, t=k, criterion="maxclust")
        # Calcolo del silhouette score usando la matrice delle distanze precalcolata
        score = silhouette_score(dist_matrix, labels, metric="precomputed")
        print(f"Metodo: {method:8s} | k={k} | Silhouette: {score:.3f}")

In [ ]:
## SALVATAGGIO DF CLUSTERS 
if FORMAT_SAVE == "csv":
    df_clusters.to_csv("DF_CLUSTERS.csv")
elif FORMAT_SAVE=="parquet":
    df_clusters.to_parquet("DF_CLUSTERS.parquet")
else: 
    print("No savage")

# 4. RICERCA PICCHI

In [ ]:
all_counts = [] 

for cc in df_clusters['cluster'].unique():
    clust_subdf = df_clusters[df_clusters['cluster'] == cc]
    comuni_cluster = clust_subdf['comune'].unique()
    
    ts_normalized_clust = ts_normalized[comuni_cluster]

    # Formato long per il plot cumulativo
    ts_normalized_long = (
        ts_normalized_clust.reset_index()
        .melt(id_vars="index", var_name="comune", value_name="value")
        .rename(columns={"index": "date"}) 
        .dropna() 
    )

    fig = plot_stats(ts_normalized_long)

    all_peak_dates = []
    comune_peak_counts = []   

    for comune in comuni_cluster:
        ts_comune = ts_normalized_clust[comune].dropna()
        peaks_idx, _ = find_peaks(ts_comune, distance=DISTANCE, prominence=PROMINCENCE)
        all_peak_dates.extend(ts_comune.index[peaks_idx])

        date_picchi = ts_comune.index[peaks_idx]
        valori_picchi = ts_comune.iloc[peaks_idx]

        comune_peak_counts.append({
            'comune': comune,
            'cluster': cc,
            'n_picchi': len(peaks_idx),
        })

        fig.add_trace(go.Scatter(
            x=date_picchi,
            y=valori_picchi,
            mode='markers',
            name=f'Picchi {comune}',
            marker=dict(size=5, symbol='circle', color="blue"),
            showlegend=False,  
            hovertemplate=f"<b>Picco ({comune})</b><br>Data: %{{x|%Y-%m-%d}}<br>Valore Norm: %{{y:.2f}}<extra></extra>"
        ))

    df_n_picchi = pd.DataFrame(comune_peak_counts).sort_values('n_picchi', ascending=False)
    all_counts.append(df_n_picchi)
    print(f"\n--- Cluster {cc}: picchi per comune ---")
    print(df_n_picchi.to_string(index=False))

    df_peaks = pd.DataFrame({'date': all_peak_dates}) if all_peak_dates else pd.DataFrame(columns=['date'])

    fig.update_layout(
        title=f"Cluster {cc} - Trend Normalizzato e Picchi Principali (Rolling {ROLLING} gg)",
        yaxis_title="Presenze Normalizzate (0=Min, 1=Max)"
    )
    
    if not df_peaks.empty:
        for d in df_peaks['date'].unique():
            fig.add_vline(
                x=d, 
                line_dash="dot", 
                line_color="blue", 
                line_width=1,
                opacity=0.3 
            )
    fig.show()

    if not df_peaks.empty:
        fig_hist = px.histogram(
            df_peaks, 
            x='date', 
            nbins=120,  
            title=f"Cluster {cc} - Concentrazione delle Date di Picco tra i Comuni",
            labels={'date': 'Data', 'count': 'N° Comuni in Picco'},
            template='plotly_white'
        )
        fig_hist.update_traces(marker_color='#1f77b4')
        fig_hist.update_layout(bargap=0.1)
        fig_hist.show()

# Analisi settimanale per cluster

In [ ]:
long = ts_normalized.reset_index().melt(id_vars="index", var_name="comune", value_name="value").rename(columns={"index": "date"})
long = long.merge(df_clusters, on="comune")
long["week"] = long["date"].dt.isocalendar().week
long["year"] = long["date"].dt.year
prof = (long.groupby(["cluster", "year", "week"])["value"].mean().reset_index())

fig = px.line(prof, x="week", y="value", color="year", facet_row="cluster", title="Profilo medio settimanale per Cluster")
fig.show()